# Decoding as inference — interactive companion

Companion to [Post 3a: Decoding as Inference](../posts/03a-decoding-as-inference.qmd).

Once you have a trained LM, the choice of *how* to sample tokens shapes
everything downstream. This notebook lets you tweak temperature, top-k,
top-p, beam width, and the number of self-consistency samples on a tiny
synthetic LM where you can see exact behavior.

**You'll do (~20 minutes):**
1. Watch the next-token distribution change with temperature.
2. See top-k vs top-p truncate the long tail differently.
3. Reproduce the MAP collapse: beam search picks a bad sequence.
4. Apply self-consistency and watch accuracy scale with N.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.decoding import (
    TinyMarkovLM, beam_search, all_beams, self_consistency,
)
from nano_agents.decoding.tinylm import _apply_top_k_top_p

rng = np.random.default_rng(0)

## 1. The tiny LM

We use a Markov-bigram LM over 8 tokens. A few "good" content sequences
get extra reward; the LM has been trained to favor them but not perfectly.
The space is small enough that we can enumerate all `7^4 = 2401` content
sequences and compute exact properties.

In [ ]:
lm = TinyMarkovLM(vocab_size=8, seq_length=4, noise=0.8, seed=2)
print(f"V = {lm.V}, L = {lm.L}")
print(f"good sequences (reward=1): {lm.good_sequences}")

# Enumerate the whole space.
seqs = lm.all_sequences()
print(f"\n{len(seqs)} total sequences")

# Top-5 by log-probability.
lps = sorted(((lm.sequence_log_prob(s), s) for s in seqs), reverse=True)
print("\nTop 5 by log-probability:")
for lp, s in lps[:5]:
    print(f"  log p = {lp:7.3f}, seq = {s}, reward = {lm.reward(s)}")

Notice already: the highest-log-prob sequence has reward 0.2, NOT 1.0.
This is the MAP collapse. Beam search will reproduce it.

### Try this
- Change the seed to 0. Does the MAP still collapse?
- Increase `noise` to 1.2. What changes about the top sequences?

## 2. Temperature

The most familiar knob. Look at what it does to a conditional distribution.

In [ ]:
# Pick a context with an interesting distribution.
cur = lm.BOS  # try changing this to 2, 3, 5
fig, ax = plt.subplots()
for T in [0.5, 1.0, 2.0, 5.0]:
    p = lm.conditional_probs(cur, T=T)
    ax.plot(np.arange(lm.V), p, "o-", label=f"T = {T}", markersize=8)
ax.set_xticks(np.arange(lm.V)); ax.set_xticklabels([f"t{i}" for i in range(lm.V)])
ax.set_xlabel("token"); ax.set_ylabel(f"P(token | t{cur})")
ax.set_title(f"Next-token distribution from t{cur} at different temperatures")
ax.legend(); ax.grid(alpha=0.3); plt.show()

### Try this
- Change `cur` to a different token. The distribution shapes vary by context.
- Add T = 0.1 to the list. The distribution becomes nearly deterministic.
- Add T = 50.0. It approaches uniform.

## 3. Top-k and top-p truncation

These remove the long tail. Look at the same context as above.

In [ ]:
cur = lm.BOS
p_full = lm.conditional_probs(cur, T=1.0)
# Sort descending for clean plotting.
order = np.argsort(p_full)[::-1]
ps_sorted = p_full[order]
labels = [f"r{i}" for i in range(lm.V)]

fig, ax = plt.subplots()
x = np.arange(lm.V)
ax.bar(x - 0.27, ps_sorted, width=0.25, label="original",
        color="#888")
p_k = _apply_top_k_top_p(p_full, top_k=3, top_p=None)
ax.bar(x, p_k[order], width=0.25, label="top-k = 3", color="#3a7ebf")
p_p = _apply_top_k_top_p(p_full, top_k=None, top_p=0.9)
ax.bar(x + 0.27, p_p[order], width=0.25, label="top-p = 0.9",
        color="#c44e52")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel("token (sorted by probability)")
ax.set_ylabel("probability after truncation + renormalize")
ax.set_title("Top-k vs top-p truncation")
ax.legend(); ax.grid(alpha=0.3, axis="y"); plt.show()

Top-p adapts to the distribution shape; top-k uses a fixed cardinality.
For this peaked distribution, top-p = 0.9 keeps more tokens than top-k = 3.

### Try this
- Set top-p = 0.5. How many tokens survive?
- Set top-k = 1. The result is greedy: a single token with mass 1.

## 4. The MAP collapse — beam search

Now we use beam search and find that high-log-prob is *not* high-reward.

In [ ]:
# Compare three strategies.
print(f"Good sequences: {lm.good_sequences}")
print()
for B in [1, 3, 10, 50]:
    seq = beam_search(lm, beam_width=B)
    r = lm.reward(seq)
    print(f"  beam-{B:2d}: seq = {seq}, reward = {r}")

# Sample at T=1 for comparison.
print("\nSamples at T=1:")
rng = np.random.default_rng(0)
for _ in range(5):
    s = lm.sample_sequence(rng, T=1.0)
    print(f"  sample: {s}, reward = {lm.reward(s)}")

Beam search of every width gets reward 0.2. It's locked in the MAP.
Sampling, by contrast, occasionally hits reward 1.0 — that's what
self-consistency will exploit next.

### Try this
- Run the second loop with `T=0.5`. Does sampling still find reward-1 sequences?
- Run with `T=2.0`. The samples are noisier — more variance, fewer hits on reward-1.

## 5. Self-consistency

Sample N times, vote on the final token. Watch accuracy scale with N.

In [ ]:
# Use a different LM where final-token answers vary - this is the
# "reasoning" structure where self-consistency really shines.
lm2 = TinyMarkovLM(vocab_size=8, seq_length=5, noise=0.6, seed=4)
correct_answers = {seq[-1] for seq in lm2.good_sequences}
print(f"Correct answer tokens: {correct_answers}")

def answer_of(seq):
    return seq[-1]

# Run the experiment.
N_values = [1, 2, 4, 8, 16, 32, 64]
n_trials = 200
accs_T1 = []
for N in N_values:
    n_correct = 0
    for trial in range(n_trials):
        r = np.random.default_rng(trial)
        votes = Counter(answer_of(lm2.sample_sequence(r, T=1.0))
                          for _ in range(N))
        winner = max(votes.keys(), key=lambda k: votes[k])
        if winner in correct_answers:
            n_correct += 1
    accs_T1.append(n_correct / n_trials)
    print(f"  N = {N:3d}: accuracy = {accs_T1[-1]:.2f}")

plt.plot(N_values, accs_T1, "o-", linewidth=2, markersize=10,
          color="#3a7ebf")
plt.axhline(1.0, color="#222", linestyle="--", alpha=0.4)
plt.xscale("log", base=2)
plt.xticks(N_values, [str(n) for n in N_values])
plt.xlabel("number of samples N"); plt.ylabel("voting accuracy")
plt.title(f"Self-consistency at T = 1.0 ({n_trials} trials per point)")
plt.grid(which="both", alpha=0.3); plt.show()

Accuracy climbs with N. At N=1 we're roughly at single-sample accuracy;
by N=64 we're near 1.0. This is **test-time compute scaling** in its
simplest form.

### Try this
- Run the same loop at `T=0.5`. Accuracy at N=1 is higher (we follow the
  MAP more closely). But how high does self-consistency reach?
- Run at `T=2.0`. Lower N=1 accuracy, but the slope to high accuracy is
  steeper. Trade-off in action.

## What's next

You've seen the major decoding strategies from first principles:

- **Temperature** scales the distribution
- **Top-k / top-p** truncate the long tail
- **Beam search** approximately maxes log-prob — and gets stuck at MAP
- **Self-consistency** marginalizes via Monte Carlo

The next post — [Post 3b: Tool use as actions](../posts/03b-tool-use.qmd) —
extends the loop one more step: the LM can now call external tools, which
*verify* or *modify* its output. That turns one-shot inference into an
agentic loop, and the techniques here become the inner machinery.